# Método Cuasi-Newton (BFGS) aplicado a SINGX

## Idea básica

El método cuasi-Newton sustituye la inversa de la Hessiana $[\nabla^2 f(x_k)]^{-1}$ por una aproximación $B_k$ simétrica y definida positiva, actualizada en cada iteración mediante una fórmula que solo usa gradientes y desplazamientos. Con
$$s_k=x_{k+1}-x_k,\qquad y_k=g_{k+1}-g_k,\qquad \rho_k=\frac{1}{y_k^t s_k},$$
la iteración es: (a) $d_k=-B_k g_k$; (b) búsqueda lineal sobre $d_k$ que satisfaga las condiciones de Wolfe; (c) actualización BFGS:
$$B_{k+1}=(I-\rho_k s_k y_k^t)\,B_k\,(I-\rho_k y_k s_k^t)+\rho_k s_k s_k^t.$$
Se toma $B_0=I$, con lo que la primera iteración coincide con un paso de máximo descenso escalado.

## Gradiente y Hessiana de SINGX

La Hessiana es **singular** en $x^*=0$ (rango $2m<n$). Por eso BFGS es especialmente adecuado: al mantener $B_k$ definida positiva, evita la inversión de una matriz singular.

Para el cálculo detallado del gradiente ver el archivo `01_max_descenso_SINGX.ipynb` y para la hessiana `02_newton_SINGX.ipynb`.

## Ejemplo de una iteración ($n=4$, $x_0=(1,1,1,1)^t$)

- $u=(11,0,-1,0)$, $f(x_0)=122$.
- $\nabla f(x_0)=(22,216,8,0)^t$, $\|\nabla f(x_0)\|\approx 217.26$.
- $\nabla^2 f(x_0)$ tiene rango $3$ (singular); el método de Newton puro falla aquí, pero BFGS no (usa $B_0=I$).
- $d_0=-B_0 g_0=(-22,-216,-8,0)^t$.
- La búsqueda lineal de Armijo/Wolfe reduce $\alpha_0$ desde $1$ hasta que se satisface la condición (para esta función, $\alpha_0\approx 7.8\times 10^{-3}$ en la primera iteración).
- Se obtiene $x_1$, se calculan $s_0=x_1-x_0$ y $y_0=g_1-g_0$, y se actualiza $B_1$ mediante la fórmula BFGS.

## Condiciones de aplicabilidad

1. $f\in C^1(\mathbb{R}^n)$: se cumple ($f$ es polinomio, $C^\infty$).
2. $B_0$ simétrica y definida positiva: $B_0=I$ cumple.
3. Búsqueda lineal que garantice $y_k^t s_k>0$ (condición de curvatura de Wolfe con $0<c_1<c_2<1$): preserva la definición positiva de $B_k$.
4. La dirección $d_k=-B_k g_k$ satisface $g_k^t d_k=-g_k^t B_k g_k<0$: es dirección de descenso.

In [1]:
# Imports
import sys
from pathlib import Path

# Sube un nivel desde la carpeta del notebook hasta la raíz del repositorio
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
from funciones import singx_f, singx_grad, singx_hess, bfgs

## Método de Newton para SINGX

El algoritmo se implementa numéricamente resolviendo el sistema lineal de Newton en cada iteración. Dado $x_0\in\mathbb{R}^n$ y tolerancia $\varepsilon>0$:

1. Calcular $g_k=\nabla f(x_k)$. Si $\|g_k\|<\varepsilon$, parar.
2. Resolver el sistema $\nabla^2 f(x_k)\,d_k = -g_k$ para obtener la dirección de Newton $d_k$.
3. Actualizar $x_{k+1}=x_k+d_k$ (paso unitario, $\alpha_k=1$).
4. $k\leftarrow k+1$, volver al paso 1.

**Nota sobre SINGX.** Lejos de $x^*=0$, la matriz Hessiana $\nabla^2 f(x_k)$ es definida positiva, lo que garantiza que $d_k$ es dirección de descenso y la convergencia cuadrática local (Teorema 9.3.1). Como ya se vio, la Hessiana es **singular** en el óptimo $x^*=0$ (rango $2m<n$) y también en puntos donde $u_3=u_4=0$ entonces el método de Newton no está bien definido. Por ello, en la práctica se parte de un punto inicial con Hessiana no singular (por ejemplo $x_0=(1,1,1,0)^t$) y se detiene el método cuando la Hessiana se vuelve mal condicionada cerca de $x^*=0$, o se recurre a la regularización $[\nabla^2 f(x_k)+\varepsilon_k I]$.

In [2]:
# ----- Ejecución: BFGS sobre SINGX, n=4, x0=(1,1,1,1) -----
x0 = np.array([1., 1., 1., 1.])

print(f"f(x0)       = {singx_f(x0):.4f}")        # esperado: 122.0000
print(f"grad(x0)    = {singx_grad(x0)}")          # esperado: [22. 216.   8.   0.]
print(f"||grad(x0)||= {np.linalg.norm(singx_grad(x0)):.4f}")  # esperado: ≈ 217.2630
print()

x_opt, hist = bfgs(singx_f, singx_grad, x0, tol=1e-10, max_iter=2000,
                   verbose=True, print_every=20)

print(f"\n--- Resultado final ---")
print(f"x_opt       = {x_opt}")                 # esperado: → (0,0,0,0) (o muy cerca)
print(f"f(x_opt)    = {singx_f(x_opt):.3e}")   # esperado: → 0 (< 1e-15)
print(f"||grad||    = {np.linalg.norm(singx_grad(x_opt)):.3e}")
print(f"Iteraciones = {len(hist['f'])-1}")      # esperado: decenas a centenas
print(f"f tras 1ª iteración = {hist['f'][1]:.4e}")  # esperado: mucho menor que 122

f(x0)       = 122.0000
grad(x0)    = [ 22. 216.   8.  -0.]
||grad(x0)||= 217.2648

k=   0  f=7.971065e+01  ||g||=2.314e+02  alpha=7.812e-03
k=   1  f=6.976482e+01  ||g||=1.742e+02  alpha=7.812e-03
k=   2  f=5.325493e+01  ||g||=1.464e+02  alpha=2.500e-01
k=   3  f=3.207452e+01  ||g||=5.301e+01  alpha=1.250e-01
k=   4  f=1.444642e+01  ||g||=6.579e+01  alpha=5.000e-01
k=  20  f=2.225598e-04  ||g||=1.812e-02  alpha=1.000e+00
k=  40  f=2.080294e-12  ||g||=1.615e-08  alpha=1.000e+00
k=  60  f=2.074466e-12  ||g||=1.676e-08  alpha=1.000e+00
k=  80  f=2.068896e-12  ||g||=1.673e-08  alpha=1.000e+00
k= 100  f=2.063349e-12  ||g||=1.670e-08  alpha=1.000e+00
k= 120  f=2.057824e-12  ||g||=1.666e-08  alpha=1.000e+00
k= 140  f=2.052322e-12  ||g||=1.663e-08  alpha=1.000e+00
k= 160  f=2.046841e-12  ||g||=1.660e-08  alpha=1.000e+00
k= 180  f=2.041382e-12  ||g||=1.656e-08  alpha=1.000e+00
k= 200  f=2.035945e-12  ||g||=1.653e-08  alpha=1.000e+00
k= 220  f=2.030530e-12  ||g||=1.650e-08  alpha=1.000e+00
k= 24